# English → Nepali Transformer — Inference

Load the trained checkpoint and translate English sentences to Nepali.

Each cell does one job:
1. **Setup** — imports, device, paths
2. **Tokenizers** — load the English / Nepali SentencePiece BPE models
3. **Model** — rebuild the `Transformer` from the checkpoint `config` and load weights
4. **Translate fn** — thin wrapper over `greedy_translate` (handles all preprocessing: encode → `</s>`, greedy decode, detokenize)
5. **Translate** — run a list of example sentences
6. **Interactive** — type your own sentence

> Run top to bottom. The notebook lives in `transformer_core/` so the local `transformer` and `data_pipeline` modules import directly.

## 1. Setup

In [1]:
import os
import sys
import torch

# Make the local modules importable regardless of where Jupyter was launched
HERE = os.path.dirname(os.path.abspath('inference.ipynb'))
if HERE not in sys.path:
    sys.path.insert(0, HERE)

from transformer import Transformer
from data_pipeline import load_tokenizers, greedy_translate

CHECKPOINT = 'models/en_ne_best.pth'

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

Device: mps


## 2. Load tokenizers

`load_tokenizers()` returns the source (English) and target (Nepali) SentencePiece processors from `tokenizers/`.

In [2]:
sp_src, sp_tgt = load_tokenizers()
print(f'English vocab : {sp_src.get_piece_size():,}')
print(f'Nepali  vocab : {sp_tgt.get_piece_size():,}')

English vocab : 16,000
Nepali  vocab : 16,000


## 3. Build model from checkpoint

The checkpoint stores the architecture `config`, so the model is rebuilt with the exact dimensions it was trained with — no hard-coded hyperparameters here.

In [3]:
if not os.path.exists(CHECKPOINT):
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT}')

ckpt   = torch.load(CHECKPOINT, map_location=device)
config = ckpt['config']
print(f'Config: {config}')

model = Transformer(
    **config,
    src_vocab_size=sp_src.get_piece_size(),
    tgt_vocab_size=sp_tgt.get_piece_size(),
)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device)
model.eval()

epoch    = ckpt.get('epoch', '?')
dev_loss = ckpt.get('dev_loss', float('nan'))
print(f'Loaded — epoch {epoch}, dev loss {dev_loss:.4f}')

Config: {'d_model': 384, 'ffn_hidden': 1536, 'num_heads': 8, 'drop_prob': 0.1, 'num_layers': 4, 'max_sequence_length': 128}
Loaded — epoch 5, dev loss 4.5089


## 4. Translate function

`greedy_translate` already does the full pipeline: SentencePiece-encode the English string, append `</s>`, run the encoder once, greedily decode target ids until `</s>`, then detokenize back to Nepali text. This wrapper just gives a clean call site.

In [4]:
def translate(sentence: str) -> str:
    """English string → Nepali string (greedy decode)."""
    return greedy_translate(model, sp_src, sp_tgt, sentence, device)

# quick smoke test
translate('Hello, how are you?')

'नमस्ते, तपाईं कसरी?'

## 5. Translate example sentences

In [5]:
examples = [
    'The weather is very nice today.',
    'My name is Ankit and I live in Kathmandu.',
    'Education is the most powerful tool to change the world.',
    'The patient had been to Nigeria, where some cases of the Ebola virus have occurred.',
    'In remote locations, without cell phone coverage, a satellite phone may be your only option.',
]

for en in examples:
    ne = translate(en)
    print(f'EN : {en}')
    print(f'NE : {ne}\n')

EN : The weather is very nice today.
NE : मौसम आज धेरै राम्रो छ।

EN : My name is Ankit and I live in Kathmandu.
NE : मेरो नाम हो, म काठमाडौंमा बस्छु ।

EN : Education is the most powerful tool to change the world.
NE : शिक्षा विश्वको परिवर्तनको लागि सबैभन्दा शक्तिशाली उपकरण हो।

EN : The patient had been to Nigeria, where some cases of the Ebola virus have occurred.
NE : धैर्यले नाइजेरियामा थियो, जहाँ इबोला भाइरसका केही केसहरू देखा पर्यो।

EN : In remote locations, without cell phone coverage, a satellite phone may be your only option.
NE : रिमोट स्थानहरूमा, सेल फोन कभरेज बिना, एक उपग्रह फोन तपाईंको मात्र विकल्प हुन सक्छ।



In [6]:
my_sentence = 'I want to learn the Nepali language.'

print('EN :', my_sentence)
print('NE :', translate(my_sentence))

EN : I want to learn the Nepali language.
NE : म नेपाली भाषा सिक्न चाहन्छु ।
